In [52]:
# imports
import kagglehub
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [53]:
# create list of csv files from dataset path
path = './datasets'
# csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
# print(csv_files)

In [54]:
# load csv
customers_df = pd.read_csv(os.path.join(path, 'olist_customers_dataset.csv'))
order_items_df = pd.read_csv(os.path.join(path, 'olist_order_items_dataset.csv'))
orders_df = pd.read_csv(os.path.join(path, 'olist_orders_dataset.csv'))
order_payments_df = pd.read_csv(os.path.join(path, 'olist_order_payments_dataset.csv'))
products_df = pd.read_csv(os.path.join(path, 'olist_products_dataset.csv'))
sellers_df = pd.read_csv(os.path.join(path, 'olist_sellers_dataset.csv'))
product_category_name_translation_df = pd.read_csv(os.path.join(path, 'product_category_name_translation.csv'))
geo_df = pd.read_csv(os.path.join(path, 'olist_geolocation_dataset.csv'))
order_reviews_df = pd.read_csv(os.path.join(path, 'olist_order_reviews_dataset.csv'))

In [55]:

print(orders_df.isna().sum())
orders_df[orders_df['order_delivered_customer_date'].isna()]['order_status'].value_counts()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

### Calculate order specifics and revenue from order_items_df

In [56]:
# calculate order revenue, products etc. 
items_agg = order_items_df.groupby('order_id').agg(
    total_items=('order_item_id', 'count'),
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
    unique_sellers=('seller_id', 'nunique'),
    unique_poroducts=('product_id', 'nunique')
).reset_index()

items_agg['total_revenue'] = items_agg['total_price'] + items_agg['total_freight']
items_agg.head(5)

,order_id,total_items,total_price,total_freight,unique_sellers,unique_poroducts,total_revenue
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,1,1,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,1,1,259.83
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,1,1,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,1,1,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,1,1,218.04


### Calculate payments per order

In [24]:
order_payments_df.head(5)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [57]:
payments_agg = order_payments_df.groupby('order_id').agg(
    num_payment_methods=('payment_type', 'nunique'),
    max_installments=('payment_installments', 'max'),
    total_payment=('payment_value', 'sum'),
    num_payment_sequence=('payment_sequential', 'max'),
    payment_types=('payment_type', lambda x: ','.join(sorted(x.unique())))
).reset_index()

conditions = [payments_agg['max_installments'] > 1, payments_agg['max_installments'].isna()]
choices = [True, pd.NA]

payments_agg['is_bnpl'] = np.select(conditions, choices, default=False)

payments_agg['is_bnpl'] = payments_agg['is_bnpl'].astype('boolean')
payments_agg.head(5)
# payments_agg['is_bnpl'].dtype

,order_id,num_payment_methods,max_installments,total_payment,num_payment_sequence,payment_types,is_bnpl
0,00010242fe8c5a6d1ba2dd792cb16214,1,2,72.19,1,credit_card,True
1,00018f77f2f0320c557190d7a144bdd3,1,3,259.83,1,credit_card,True
2,000229ec398224ef6ca0657da4fc703e,1,5,216.87,1,credit_card,True
3,00024acbcdf0a6daa1e931b038114c75,1,2,25.78,1,credit_card,True
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,3,218.04,1,credit_card,True


### Prepare order reviews and customer tables for join

In [58]:
# prepare avg score per order
order_avg_review = order_reviews_df.groupby('order_id')['review_score'].mean().reset_index()

order_avg_review.head(5)

,order_id,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,5.0
1,00018f77f2f0320c557190d7a144bdd3,4.0
2,000229ec398224ef6ca0657da4fc703e,5.0
3,00024acbcdf0a6daa1e931b038114c75,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0


In [59]:
# prepare customers table for merge
customer_info = customers_df[['customer_id', 'customer_unique_id', 'customer_city', 'customer_state']]

### Create Master Orders Table

In [61]:
# merge all 4 tables on orders_df
master_orders = (
    orders_df
    .merge(customer_info, on='customer_id', how='left')
    .merge(order_avg_review, on='order_id', how='left')
    .merge(items_agg, on='order_id', how='left')
    .merge(payments_agg, on='order_id', how='left')
)

master_orders.shape

(99441, 24)

In [67]:
master_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_city,...,total_freight,unique_sellers,unique_poroducts,total_revenue,num_payment_methods,max_installments,total_payment,num_payment_sequence,payment_types,is_bnpl
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,sao paulo,...,8.72,1.0,1.0,38.71,2.0,1.0,38.71,3.0,"credit_card,voucher",False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,barreiras,...,22.76,1.0,1.0,141.46,1.0,1.0,141.46,1.0,boleto,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,...,19.22,1.0,1.0,179.12,1.0,3.0,179.12,1.0,credit_card,True
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,...,27.20,1.0,1.0,72.20,1.0,1.0,72.20,1.0,credit_card,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,...,8.72,1.0,1.0,28.62,1.0,1.0,28.62,1.0,credit_card,False
